# Tableau Calculated Fields

Up until this point, we have been using Python to manipulate our data. But in the real world, Data Scientists often have to build dashboards in enterprise Business Intelligence (BI) tools like **Tableau** or **PowerBI**. 

While these tools are visual and drag-and-drop, they have their own underlying programming languages. If you want to build advanced dashboards, you must learn how to write **Calculated Fields**. In this lesson, we will learn the core logic of Tableau calculations and compare them directly to the Pandas code you already know!

A Calculated Field in Tableau is exactly like Feature Engineering in Python. It allows you to create new data from data that already exists in your dataset.

However, Tableau calculates data in two distinctly different ways: **Row-Level** and **Aggregate**. Understanding the difference between these two is the single biggest hurdle for beginners.

Let's set up a Python sandbox with a simple Retail dataset to demonstrate how Tableau's calculation engine works behind the scenes.

In [1]:
import pandas as pd
import numpy as np

# Create a sample retail dataset
data = {
    'Order_ID': ['A1', 'A2', 'A3', 'B1', 'B2'],
    'Region': ['East', 'East', 'East', 'West', 'West'],
    'Sales': [100, 200, 300, 400, 500],
    'Cost': [80, 100, 250, 200, 400]
}

df = pd.DataFrame(data)

print("✅ Retail Dataset Loaded!")
display(df)

✅ Retail Dataset Loaded!


,Order_ID,Region,Sales,Cost
0,A1,East,100,80
1,A2,East,200,100
2,A3,East,300,250
3,B1,West,400,200
4,B2,West,500,400


# 1. Row-Level Calculations
A Row-Level calculation computes the math for *every single row independently*, before the dashboard even tries to group or chart the data. 

* **The Goal**: Calculate the exact Profit for each individual order.
* **Tableau Syntax**: `[Sales] - [Cost]`

In [2]:
# Create a copy for our row-level math
df_row = df.copy()

# The Pandas Equivalent of a Tableau Row-Level Calculation:
df_row['Profit (Row-Level)'] = df_row['Sales'] - df_row['Cost']

print("--- Row-Level Calculation ---")
display(df_row)

--- Row-Level Calculation ---


,Order_ID,Region,Sales,Cost,Profit (Row-Level)
0,A1,East,100,80,20
1,A2,East,200,100,100
2,A3,East,300,250,50
3,B1,West,400,200,200
4,B2,West,500,400,100


# 2. Aggregate Calculations
This is where Tableau tricks people. What if we want to know our overall **Profit Margin** (Profit divided by Sales) for each Region?

If you write `[Profit] / [Sales]` in Tableau, it will calculate the margin for Row 1, then the margin for Row 2, and then add those margins together. Adding percentages together is a massive mathematical error!

Instead, you must use an **Aggregate Calculation**. You tell Tableau to add up all the Sales first, add up all the Profits first, and *then* divide them.

* **The Goal**: Calculate the True Profit Margin per Region.
* **Tableau Syntax**: `SUM([Profit]) / SUM([Sales])`

In [3]:
# The Pandas Equivalent of a Tableau Aggregate Calculation:

# 1. First, we group by Region (Tableau does this automatically when you drag 'Region' to a chart)
region_agg = df_row.groupby('Region').agg({
    'Sales': 'sum',
    'Profit (Row-Level)': 'sum'
}).reset_index()

# 2. THEN, we perform the division on the aggregated numbers
region_agg['True_Margin (Aggregated)'] = region_agg['Profit (Row-Level)'] / region_agg['Sales']

print("--- Aggregate Calculation ---")
display(region_agg)

--- Aggregate Calculation ---


,Region,Sales,Profit (Row-Level),True_Margin (Aggregated)
0,East,600,170,0.283333
1,West,900,300,0.333333


*(Insight: If an executive asks for a ratio or a percentage in a dashboard, 99% of the time you must use an Aggregate calculation like `SUM(A) / SUM(B)`!)*

# 3. Logical Functions (IF / THEN)
Just like in Python, you can write `IF/ELSE` statements in Tableau to categorize your data dynamically. This is incredibly useful for creating custom color-coding rules in your dashboards.

* **The Goal**: Flag any individual order that has over \$350 in sales as a "High Value" order.
* **Tableau Syntax**: 
  ```text
  IF [Sales] > 350 THEN 
      'High Value' 
  ELSEIF [Sales] > 150 THEN 
      'Medium Value'
  ELSE 
      'Low Value' 
  END
  ```

In [4]:
# The Pandas Equivalent using numpy.select
conditions = [
    df_row['Sales'] > 350,
    df_row['Sales'] > 150
]
choices = ['High Value', 'Medium Value']

# default='Low Value' acts as the ELSE statement
df_row['Order_Category'] = np.select(conditions, choices, default='Low Value')

print("--- Logical (IF/THEN) Calculation ---")
display(df_row[['Order_ID', 'Sales', 'Order_Category']])

--- Logical (IF/THEN) Calculation ---


,Order_ID,Sales,Order_Category
0,A1,100,Low Value
1,A2,200,Medium Value
2,A3,300,Medium Value
3,B1,400,High Value
4,B2,500,High Value


# 4. Table Calculations (Window Functions)
The most advanced calculations in Tableau are called **Table Calculations**. These do not look at the underlying raw data. Instead, they look at the *visual table you just built on the screen* and perform math on it. 

The most common example is a **Running Total** (Cumulative Sum).

* **The Goal**: Show how our total sales grew order-by-order.
* **Tableau Syntax**: `RUNNING_SUM(SUM([Sales]))`

In [5]:
# The Pandas Equivalent of a Tableau Table Calculation

# We use .cumsum() to calculate the running total
df_row['Running_Total_Sales'] = df_row['Sales'].cumsum()

print("--- Table Calculation (Running Total) ---")
display(df_row[['Order_ID', 'Sales', 'Running_Total_Sales']])

--- Table Calculation (Running Total) ---


,Order_ID,Sales,Running_Total_Sales
0,A1,100,100
1,A2,200,300
2,A3,300,600
3,B1,400,1000
4,B2,500,1500


*(Insight: In Tableau, you usually don't even have to write the code for Table Calculations. You can right-click any pill in your view and select "Quick Table Calculation -> Running Total"!)*

---

## Real-World Use Case or Analogy:
Think of Tableau's Calculation engine like **Managing a Corporate Budget**:

* **Row-Level (The Receipts)**: An employee submits an expense report. You look at a single receipt from an office supply store: $50 for paper, minus a $10 coupon, equals $40 paid. You calculate this independently for every single receipt in the stack.
* **Aggregate (The Department Budget)**: The CEO asks, "What percentage of the Marketing Department's budget was spent on advertising?" You cannot look at individual receipts for this. You must gather *all* the Marketing receipts into a pile (SUM), gather the *entire* budget into a pile (SUM), and then divide the two massive piles.
* **Table Calculation (The Whiteboard)**: You have written the total spend for Jan, Feb, and Mar on a whiteboard. The CEO asks, "What is our running total for the quarter?" You don't go back into the filing cabinet to look at the receipts (the raw data). You just look at the three numbers on the whiteboard (the visual table), add them together in your head, and write the answer. 

---